In [1]:
import pandas as pd
import warnings 
warnings.filterwarnings("ignore")

# Control Variable(고객데이터) 추가하기
1. 데이터 호출
2. 기본 고객정보 코딩
2. 데이터 결합


### 1. 데이터 호출
* 분석용데이터, 고객정보, 1년간고객구매이력
* 데이터 결합을 위한 INCS_NO 필요   

### 2. 고객 기본정보 코딩
* 남:0, 여:1
* age - 10,20대, 30대, 40대, 50대, 60대이상
* groupby를 이용해 고객별 연평균구매액 코딩
* 고객정보 + 구매이력 결합

### 3. 데이터 결합
* 분석용데이터 + 고객정보 결합
* join 후 누락된 INCS_NO가 많은지 확인 필요
* 누락된 데이터가 많다면 데이터 새로 다운받아야 할 듯 함

### 1. 데이터 호출

In [5]:
df = pd.read_csv('./data/sess_indicate_cutoff4.csv', index_col=0)
print(df.shape)
df.head(2)

(351657, 10)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off
1,000117b4880909a482cbfb7a65ddf0f71722298631,22,6,1,5,22,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 09:19:43.659,26.250000,0
2,000117b4880909a482cbfb7a65ddf0f71722301833,51,13,3,23,51,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 10:29:11.286,44.677419,0


In [6]:
cust = pd.read_csv('./raw/CUST_INFO.csv')
print(cust.shape)
cust.head(2)

(966941, 3)


,INCS_NO,AGE,SEX_CD
0,e83ec27d1ea0bdfbee8296ca86cab339fa38be7e704145...,33,F
1,5888134982e132a44ca560752ebdb604521f71573c8466...,67,F


In [7]:
cust_sal = pd.read_csv('./raw/CUST_SAL.csv')
cust_sal['STND_YMD'] = pd.to_datetime(cust_sal['STND_YMD'])
print(cust_sal.shape)
cust_sal.head(2)

(2868414, 3)


,INCS_NO,POS_NET_PRD_SAL_AMT,STND_YMD
0,857c1e5d32cac4c06011e4c9297e7cd05f6f8629288bd1...,32850.0,2024-02-21
1,2df11ad7837d63b48fed4bd6db79a932d74239c703f9b1...,23800.0,2024-02-21


### 2. 고객 기본정보 코딩

In [8]:
cust_sal['YEAR'] = cust_sal['STND_YMD'].dt.year 
cust_sal_avg = cust_sal.groupby(['INCS_NO', 'YEAR']).POS_NET_PRD_SAL_AMT.mean().reset_index() 
cust_sal_avg = cust_sal_avg.rename(columns={'POS_NET_PRD_SAL_AMT': 'AVG_SAL_AMT'})
cust_sal_avg = cust_sal_avg.groupby(['INCS_NO']).AVG_SAL_AMT.mean().reset_index()

cust_sal_avg.head(2)

,INCS_NO,AVG_SAL_AMT
0,0000a9677f34fb28ae920f7346600a59c9ef3e2a65c347...,0.000000
1,0000e30a4703362d8d858cf0b777871baa3c392562f3d1...,5566.666667


In [9]:
cust = pd.merge(cust, cust_sal_avg, how='left', on='INCS_NO')
print(cust.shape)
cust.head(2)

(966941, 4)


,INCS_NO,AGE,SEX_CD,AVG_SAL_AMT
0,e83ec27d1ea0bdfbee8296ca86cab339fa38be7e704145...,33,F,NaN
1,5888134982e132a44ca560752ebdb604521f71573c8466...,67,F,NaN


In [10]:
cust[cust['AVG_SAL_AMT'].notna()].shape

(253882, 4)

### 3. 데이터 결합

In [11]:
temp = pd.merge(df, cust, how='left', on='INCS_NO')
print(temp.shape)
temp.head(2)

(351657, 13)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off,AGE,SEX_CD,AVG_SAL_AMT
0,000117b4880909a482cbfb7a65ddf0f71722298631,22,6,1,5,22,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 09:19:43.659,26.250000,0,36.0,F,7695.454545
1,000117b4880909a482cbfb7a65ddf0f71722301833,51,13,3,23,51,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 10:29:11.286,44.677419,0,36.0,F,7695.454545


In [12]:
# cust가 누락된 데이터 수 확인
# 누락된 데이터가 너무 많다면 cust 데이터 재검토 필요

print(temp[temp['AGE'].isna()].shape)
print(temp[temp['AVG_SAL_AMT'].isna()].shape)

(14, 13)
(58625, 13)


In [13]:
temp = temp[temp['AGE'].notna()]
cv_df = temp[temp['AVG_SAL_AMT'].notna()]
print(cv_df.shape)
cv_df.head(2)

(293032, 13)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off,AGE,SEX_CD,AVG_SAL_AMT
0,000117b4880909a482cbfb7a65ddf0f71722298631,22,6,1,5,22,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 09:19:43.659,26.250000,0,36.0,F,7695.454545
1,000117b4880909a482cbfb7a65ddf0f71722301833,51,13,3,23,51,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 10:29:11.286,44.677419,0,36.0,F,7695.454545


In [14]:
cv_df['INCS_NO'].nunique()

57249

In [15]:
def age_coding(age):
    if age < 30:
        return 1020
    elif age < 40:
        return 30
    elif age < 50:
        return 40
    elif age < 60:
        return 50
    else:
        return 60

def sex_coding(sex):
    if sex == 'F':
        return 1
    else:
        return 0

In [16]:
# 나이 원핫인코딩
dummy_df = cv_df.copy()
dummy_df['AGE'] = dummy_df['AGE'].apply(age_coding)
dummy_df['SEX_CD'] = dummy_df['SEX_CD'].apply(sex_coding)
dummy_df = pd.get_dummies(dummy_df, columns=['AGE'], dtype=int)

# 40대 데이터 제거
dummy_df = dummy_df.drop(columns=['AGE_40'])
print(dummy_df.shape)
dummy_df.head(2)

(293032, 16)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off,SEX_CD,AVG_SAL_AMT,AGE_30,AGE_50,AGE_60,AGE_1020
0,000117b4880909a482cbfb7a65ddf0f71722298631,22,6,1,5,22,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 09:19:43.659,26.250000,0,1,7695.454545,1,0,0,0
1,000117b4880909a482cbfb7a65ddf0f71722301833,51,13,3,23,51,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 10:29:11.286,44.677419,0,1,7695.454545,1,0,0,0


In [24]:
# 주말변수 추가



dummy_df['LST_SESS_TIME'] = pd.to_datetime(dummy_df['LST_SESS_TIME'])

dummy_df['WEEKEND'] = dummy_df['LST_SESS_TIME'].apply(lambda x: x.weekday())

dummy_df['WEEKEND'] = dummy_df['WEEKEND'].apply(lambda x: 1 if x >= 5 else 0)
dummy_df.head(2)

,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off,SEX_CD,AVG_SAL_AMT,AGE_30,AGE_50,AGE_60,AGE_1020,WEEKDAY,WEEKEND
0,000117b4880909a482cbfb7a65ddf0f71722298631,22,6,1,5,22,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 09:19:43.659,26.250000,0,1,7695.454545,1,0,0,0,1,0
1,000117b4880909a482cbfb7a65ddf0f71722301833,51,13,3,23,51,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 10:29:11.286,44.677419,0,1,7695.454545,1,0,0,0,1,0


In [25]:
dummy_df.to_csv('./data/cv_data.csv')

In [ ]:
dummy_df['INCS_NO'].nunique()

4637